# GPT Notebook

This notebook contains the standalone GPT pretraining path. 

`LayerNormalization`, `Linear`, and `GLU` are custom functions not from torch.nn

`MultiHeadAttention` computes self attention with optional causal and padding masks. 

`TransformerBlock` combines attention, normalization, and the feed forward network. 

`ConfigParametersLLM` and `OptimParameters` hold model and optimizer settings. 

`PreTrainTextDataset` creates next token prediction windows from the Tiny Shakespeare corpus. 

`load_tinyshakespeare_tokens` tokenizes the text corpus, 

`build_gpt_dataloader` wraps it in a dataloader, 

`evaluate_gpt_loss` computes mean loss over a loader. 

`GPT` builds the decoder only transformer with token embeddings, position embeddings, transformer blocks, and the output projection. The remaining cells set up training, run optimization, save the checkpoint, and plot the loss curve.

In [ ]:
import os

import matplotlib.pyplot as plt
import torch

from gpt_standalone import (
    ConfigParametersLLM,
    GPT,
    OptimParameters,
    PreTrainTextDataset,
    TransformerBlock,
    LayerNormalization,
    Linear,
    GLU,
    MultiHeadAttention,
    build_gpt_dataloader,
    evaluate_gpt_loss,
    load_tinyshakespeare_tokens,
)


## Training Setup

In [ ]:
data_path = "./char-rnn/data/tinyshakespeare/input.txt"
output_dir = "./outputs/llm"
os.makedirs(output_dir, exist_ok=True)

tokenizer, train_data = load_tinyshakespeare_tokens(data_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

llm_config = ConfigParametersLLM(vocab_size=len(tokenizer), device=device)
opt_cfg = OptimParameters()

train_loader = build_gpt_dataloader(train_data, llm_config, shuffle=True)
model = GPT(llm_config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    opt_cfg.lr,
    betas=opt_cfg.betas,
    eps=opt_cfg.eps,
)

len(train_loader)


## Training Loop

In [ ]:
model.train()
step = 0
max_steps = 100000
train_loss = []

while step < max_steps:
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        logits, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss.append(loss.item())
        step += 1

        if step % 100 == 0:
            print(f"Step {step} | Loss {loss.item():.4f}")

        if step >= max_steps:
            break

torch.save(
    {"step": step, "model": model.state_dict(), "train_loss": train_loss},
    os.path.join(output_dir, "llm_final.pt"),
)


## Loss Plot

In [ ]:
window = 100
smoothed = [
    sum(train_loss[i : i + window]) / window
    for i in range(len(train_loss) - window + 1)
]

plt.plot(train_loss, alpha=0.2, label="raw")
plt.plot(range(window - 1, len(train_loss)), smoothed, label="smoothed")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("GPT Training Loss")
plt.legend()
plt.show()
